## Dynamic Sign Recognition — Webcam Pipeline

**Architecture:**
1. **MediaPipe Hands** (Tasks API) — Continuous hand tracking + 21-keypoint extraction + bounding box
2. **Activation Trigger** — Open-palm static pose held for N frames unlocks recording
3. **InceptionV4** (timm) — Spatial feature extractor on cropped hand frames
4. **Transformer** — Temporal classifier over the collected feature sequence

## 1 · Install Dependencies

In [1]:
import kagglehub
path = kagglehub.dataset_download("soumicksarker/ipn-hand-dataset")

C:\Users\Veron Dalumpines\.virtualenvs\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
!pip install mediapipe opencv-python timm torchvision --quiet


[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2 · Imports & Configuration

In [3]:
import cv2
import mediapipe as mp
import numpy as np
import torch
import torch.nn as nn
import timm
import time
import urllib.request
from collections import deque
from pathlib import Path
from torchvision import transforms

print(f"PyTorch  : {torch.__version__}")
print(f"MediaPipe: {mp.__version__}")
print(f"Device   : {'cuda' if torch.cuda.is_available() else 'cpu'}")

PyTorch  : 2.11.0+cpu
MediaPipe: 0.10.35
Device   : cpu


In [4]:
# ── IPN Hand Dataset — 14 gesture classes ────────────────────────────────────
GESTURE_LABELS = {
    0:  'D0X — No gesture',
    1:  'D1  — Point (index)',
    2:  'D2  — Point (two fingers)',
    3:  'D3  — Click (index)',
    4:  'D4  — Click (two fingers)',
    5:  'D5  — Throw up',
    6:  'D6  — Throw down',
    7:  'D7  — Throw left',
    8:  'D8  — Throw right',
    9:  'D9  — Open twice',
    10: 'D10 — Double-click (index)',
    11: 'D11 — Double-click (two fingers)',
    12: 'D12 — Zoom in',
    13: 'D13 — Zoom out',
}
NUM_CLASSES = len(GESTURE_LABELS)

# ── Hyper-parameters ──────────────────────────────────────────────────────────
class Cfg:
    CAM_ID          = 0
    CAM_W, CAM_H    = 640, 480
    TRIGGER_FRAMES  = 20       # open-palm frames required to arm
    SEQ_LEN         = 32       # frames captured per gesture
    IMG_SIZE        = (299, 299)
    FEATURE_DIM     = 1536     # InceptionV4 global-pool output
    HIDDEN_DIM      = 256
    NUM_HEADS       = 4
    NUM_LAYERS      = 2
    DROPOUT         = 0.1
    DEVICE          = 'cuda' if torch.cuda.is_available() else 'cpu'

print("Config loaded.")

Config loaded.


## 3 · MediaPipe — Tasks API (Hand Landmarker)

In [5]:
import urllib.request
from pathlib import Path
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision

# Download the hand landmarker model once (~8 MB)
MODEL_PATH = 'hand_landmarker.task'
if not Path(MODEL_PATH).exists():
    print("Downloading hand_landmarker.task (~8 MB) ...")
    urllib.request.urlretrieve(
        "https://storage.googleapis.com/mediapipe-models/"+
        "hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task",
        MODEL_PATH,
    )
    print("Done.")
else:
    print(f"Model already present at '{MODEL_PATH}'.")

# Hand connections for drawing (21 landmarks, 0-indexed)
HAND_CONNECTIONS = [
    (0, 1), (1, 2), (2, 3), (3, 4),           # Thumb
    (0, 5), (5, 6), (6, 7), (7, 8),           # Index
    (5, 9), (9, 10), (10, 11), (11, 12),      # Middle
    (9, 13), (13, 14), (14, 15), (15, 16),    # Ring
    (13, 17), (17, 18), (18, 19), (19, 20),   # Pinky
    (0, 17)                                     # Palm
]

def build_hands_tracker():
    """
    Returns a HandLandmarker in VIDEO running mode.
    VIDEO mode is stateful - it tracks across frames using timestamps,
    reducing detection latency compared to IMAGE mode.
    """
    base_opts = mp_python.BaseOptions(model_asset_path=MODEL_PATH)
    options   = mp_vision.HandLandmarkerOptions(
        base_options                  = base_opts,
        num_hands                     = 1,
        min_hand_detection_confidence = 0.70,
        min_hand_presence_confidence  = 0.55,
        min_tracking_confidence       = 0.55,
        running_mode                  = mp_vision.RunningMode.VIDEO,
    )
    return mp_vision.HandLandmarker.create_from_options(options)

def draw_hand(frame_bgr, hand_landmarks_list):
    h, w = frame_bgr.shape[:2]
    for hand_lms in hand_landmarks_list:
        points = [(int(lm.x * w), int(lm.y * h)) for lm in hand_lms]
        for connection in HAND_CONNECTIONS:
            pt1, pt2 = points[connection[0]], points[connection[1]]
            cv2.line(frame_bgr, pt1, pt2, (255, 80, 0), 2)
        for pt in points:
            cv2.circle(frame_bgr, pt, 3, (0, 255, 120), -1)

def get_hand_bbox(frame, hand_landmarks, pad=25):
    """Compute a padded bounding box from 21 NormalizedLandmark objects."""
    h, w  = frame.shape[:2]
    xs    = [lm.x * w for lm in hand_landmarks]
    ys    = [lm.y * h for lm in hand_landmarks]
    x_min = max(0, int(min(xs)) - pad)
    x_max = min(w, int(max(xs)) + pad)
    y_min = max(0, int(min(ys)) - pad)
    y_max = min(h, int(max(ys)) + pad)
    return frame[y_min:y_max, x_min:x_max], (x_min, y_min, x_max, y_max)

print("MediaPipe Tasks API helpers ready.")

Model already present at 'hand_landmarker.task'.
MediaPipe Tasks API helpers ready.


## 4 · Activation Trigger

An **open palm** held for `TRIGGER_FRAMES` consecutive frames arms the system.  
Recording begins the moment the hand moves away from the trigger pose.

In [6]:
FINGER_TIPS = [8, 12, 16, 20]   # index, middle, ring, pinky tip
FINGER_PIPS = [6, 10, 14, 18]   # PIP joints

def is_open_palm(hand_landmarks, handedness='Right'):
    """
    Returns True when 3+ fingers are extended and the thumb is abducted.
    hand_landmarks : flat list of NormalizedLandmark (Tasks API format)
    handedness     : 'Right' or 'Left' — Tasks API returns the anatomical label.
    """
    lm = hand_landmarks
    fingers_up = sum(
        lm[tip].y < lm[pip].y
        for tip, pip in zip(FINGER_TIPS, FINGER_PIPS)
    )
    thumb_up = (
        lm[4].x > lm[2].x if handedness == 'Right'
        else lm[4].x < lm[2].x
    )
    return fingers_up >= 3 and thumb_up


print("Activation trigger ready.")

Activation trigger ready.


## 5 · Spatial Extractor — InceptionV4

In [7]:
class SpatialExtractor(nn.Module):
    """
    Pre-trained InceptionV4 backbone with the classification head removed.
    Outputs a (1536,) global-average-pooled embedding per cropped hand image.
    All weights are frozen — used for inference only.
    """
    def __init__(self, device='cpu'):
        super().__init__()
        self.backbone = timm.create_model(
            'inception_v4',
            pretrained  = True,
            num_classes = 0,
        )
        self.backbone.eval()
        self.device = device
        self.to(device)
        self.preprocess = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize(Cfg.IMG_SIZE),
            transforms.ToTensor(),
            transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
        ])

    @torch.no_grad()
    def encode(self, crop_bgr: np.ndarray):
        """Encode a single BGR crop → (1536,) feature tensor on CPU."""
        if crop_bgr is None or crop_bgr.size == 0:
            return None
        rgb    = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB)
        tensor = self.preprocess(rgb).unsqueeze(0).to(self.device)
        feat   = self.backbone(tensor)
        return feat.squeeze(0).cpu()


print("Loading InceptionV4 weights (downloads once ~170 MB) ...")
spatial_extractor = SpatialExtractor(device=Cfg.DEVICE)
print(f"SpatialExtractor ready on {Cfg.DEVICE}. Output dim: {Cfg.FEATURE_DIM}")

Loading InceptionV4 weights (downloads once ~170 MB) ...
SpatialExtractor ready on cpu. Output dim: 1536


## 6 · Temporal Classifier — Transformer Encoder

In [8]:
class TemporalTransformer(nn.Module):
    """
    Classifies a fixed-length sequence of InceptionV4 embeddings.
    Input : (B, T, feature_dim)
    Output: (B, num_classes) raw logits
    """
    def __init__(
        self,
        feature_dim : int   = Cfg.FEATURE_DIM,
        num_classes : int   = NUM_CLASSES,
        hidden_dim  : int   = Cfg.HIDDEN_DIM,
        num_heads   : int   = Cfg.NUM_HEADS,
        num_layers  : int   = Cfg.NUM_LAYERS,
        dropout     : float = Cfg.DROPOUT,
        max_len     : int   = 128,
    ):
        super().__init__()
        self.input_proj = nn.Linear(feature_dim, hidden_dim)
        self.cls_token  = nn.Parameter(torch.zeros(1, 1, hidden_dim))

        # Sinusoidal positional encoding
        pe  = torch.zeros(max_len + 1, hidden_dim)
        pos = torch.arange(0, max_len + 1).unsqueeze(1).float()
        div = torch.exp(
            torch.arange(0, hidden_dim, 2).float() * -(np.log(10000.0) / hidden_dim)
        )
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim, nhead=num_heads,
            dim_feedforward=hidden_dim * 4,
            dropout=dropout, batch_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.norm        = nn.LayerNorm(hidden_dim)
        self.head        = nn.Linear(hidden_dim, num_classes)
        nn.init.xavier_uniform_(self.cls_token)

    def forward(self, x):
        B, T, _ = x.shape
        x   = self.input_proj(x)
        cls = self.cls_token.expand(B, -1, -1)
        x   = torch.cat([cls, x], dim=1) + self.pe[:, :T + 1]
        x   = self.transformer(x)
        return self.head(self.norm(x[:, 0]))


temporal_classifier = TemporalTransformer().to(Cfg.DEVICE)
print(f"TemporalTransformer params: {sum(p.numel() for p in temporal_classifier.parameters()):,}")

TemporalTransformer params: 1,977,358


## 7 · (Optional) Load Pre-trained Weights

Set `MODEL_WEIGHTS` to your `.pt` checkpoint path if you have one.  
Otherwise the pipeline runs with random temporal weights (useful for testing the full flow).

In [9]:
MODEL_WEIGHTS = None  # e.g. 'checkpoints/best_model.pt'

if MODEL_WEIGHTS:
    ckpt = torch.load(MODEL_WEIGHTS, map_location=Cfg.DEVICE)
    temporal_classifier.load_state_dict(ckpt['temporal_classifier'])
    print(f"Loaded weights from {MODEL_WEIGHTS}")
else:
    print("No weights loaded — running with random temporal classifier weights.")

temporal_classifier.eval()

No weights loaded — running with random temporal classifier weights.


TemporalTransformer(
  (input_proj): Linear(in_features=1536, out_features=256, bias=True)
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
        )
        (linear1): Linear(in_features=256, out_features=1024, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=1024, out_features=256, bias=True)
        (norm1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (norm): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
  (head): Linear(in_features=256, out_features=14, bias=True)
)

## 8 · Full Webcam Pipeline

**State machine:**
```
IDLE ──(open palm N frames)──► ARMED ──(hand moves)──► RECORDING ──(SEQ_LEN frames)──► IDLE
```

> `process()` requires a monotonically increasing `timestamp_ms` — the Tasks API  
> VIDEO mode uses it internally to maintain inter-frame tracking state.

In [10]:
class GestureState:
    IDLE      = 'IDLE'
    ARMED     = 'ARMED'
    RECORDING = 'RECORDING'


class GesturePipeline:
    def __init__(self, spatial_ext, temporal_cls, gesture_labels):
        self.spatial  = spatial_ext
        self.temporal = temporal_cls
        self.labels   = gesture_labels
        self.hands    = build_hands_tracker()
        self.state           = GestureState.IDLE
        self.trigger_count   = 0
        self.feat_buffer     = deque(maxlen=Cfg.SEQ_LEN)
        self.last_prediction = '—'
        self.last_confidence = 0.0

    def _classify(self):
        feats = torch.stack(list(self.feat_buffer)).unsqueeze(0).to(Cfg.DEVICE)
        with torch.no_grad():
            probs      = torch.softmax(self.temporal(feats), dim=-1)
            conf, pred = probs.max(dim=-1)
        return self.labels.get(pred.item(), f'Class {pred.item()}'), conf.item()

    def process(self, frame_bgr: np.ndarray, timestamp_ms: int) -> np.ndarray:
        rgb      = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
        results  = self.hands.detect_for_video(mp_image, timestamp_ms)

        bbox = None

        if results.hand_landmarks:
            hand_lm    = results.hand_landmarks[0]
            handedness = results.handedness[0][0].category_name

            draw_hand(frame_bgr, results.hand_landmarks)
            crop, bbox = get_hand_bbox(frame_bgr, hand_lm)
            triggered  = is_open_palm(hand_lm, handedness)

            if self.state == GestureState.IDLE:
                if triggered:
                    self.trigger_count += 1
                    self.state = GestureState.ARMED
                else:
                    self.trigger_count = 0

            elif self.state == GestureState.ARMED:
                if triggered:
                    self.trigger_count += 1
                    if self.trigger_count >= Cfg.TRIGGER_FRAMES:
                        self.state = GestureState.RECORDING
                        self.feat_buffer.clear()
                else:
                    self.trigger_count = 0
                    self.state = GestureState.IDLE

            elif self.state == GestureState.RECORDING:
                feat = self.spatial.encode(crop)
                if feat is not None:
                    self.feat_buffer.append(feat)
                if len(self.feat_buffer) >= Cfg.SEQ_LEN:
                    self.last_prediction, self.last_confidence = self._classify()
                    self.state         = GestureState.IDLE
                    self.trigger_count = 0
        else:
            if self.state == GestureState.RECORDING and len(self.feat_buffer) >= Cfg.SEQ_LEN // 2:
                self.last_prediction, self.last_confidence = self._classify()
            self.state         = GestureState.IDLE
            self.trigger_count = 0

        self._draw_overlay(frame_bgr, bbox)
        return frame_bgr

    def _draw_overlay(self, frame, bbox):
        h, w = frame.shape[:2]
        STATE_COLORS = {
            GestureState.IDLE:      (180, 180, 180),
            GestureState.ARMED:     (0,   200, 255),
            GestureState.RECORDING: (0,   80,  255),
        }
        color = STATE_COLORS[self.state]

        if bbox:
            x0, y0, x1, y1 = bbox
            cv2.rectangle(frame, (x0, y0), (x1, y1), color, 2)

        cv2.rectangle(frame, (0, 0), (w, 95), (20, 20, 20), -1)

        if self.state == GestureState.ARMED:
            pct = int(self.trigger_count / Cfg.TRIGGER_FRAMES * (w - 20))
            cv2.rectangle(frame, (10, 75), (10 + pct, 88), (0, 200, 255), -1)
            cv2.rectangle(frame, (10, 75), (w - 10, 88), (100, 100, 100), 1)
            label = f'ARMED  ({self.trigger_count}/{Cfg.TRIGGER_FRAMES}) — hold open palm'
        elif self.state == GestureState.RECORDING:
            pct = int(len(self.feat_buffer) / Cfg.SEQ_LEN * (w - 20))
            cv2.rectangle(frame, (10, 75), (10 + pct, 88), (0, 80, 255), -1)
            cv2.rectangle(frame, (10, 75), (w - 10, 88), (100, 100, 100), 1)
            label = f'RECORDING  ({len(self.feat_buffer)}/{Cfg.SEQ_LEN} frames)'
        else:
            label = 'IDLE — show open palm to activate'

        cv2.putText(frame, label, (10, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.65, color, 2)
        cv2.putText(frame,
            f'Gesture: {self.last_prediction}  ({self.last_confidence:.0%})',
            (10, 58), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (50, 255, 50), 2)

        if self.state == GestureState.RECORDING:
            cv2.circle(frame, (w - 25, 25), 10, (0, 0, 255), -1)
            cv2.putText(frame, 'REC', (w - 70, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 0, 255), 2)

    def close(self):
        self.hands.close()


print("GesturePipeline class ready.")

GesturePipeline class ready.


## 9 · Run — Webcam Inference

- **Show an open palm** and hold still until the progress bar fills → system arms.
- **Perform your gesture** — pipeline records `SEQ_LEN` frames automatically.
- Result is shown on screen after classification.
- Press **`q`** inside the window to quit.

In [11]:
pipeline = GesturePipeline(
    spatial_ext    = spatial_extractor,
    temporal_cls   = temporal_classifier,
    gesture_labels = GESTURE_LABELS,
)

cap = cv2.VideoCapture(Cfg.CAM_ID)
cap.set(cv2.CAP_PROP_FRAME_WIDTH,  Cfg.CAM_W)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, Cfg.CAM_H)
cap.set(cv2.CAP_PROP_BUFFERSIZE, 1)

print("Webcam opened. Press 'q' inside the window to quit.")
start_time = time.time()

try:
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            print("Camera read failed — check Cfg.CAM_ID.")
            break

        frame        = cv2.flip(frame, 1)
        timestamp_ms = int((time.time() - start_time) * 1000)   # monotonic ms
        frame        = pipeline.process(frame, timestamp_ms)

        cv2.imshow('Dynamic Sign Recognition', frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
finally:
    cap.release()
    cv2.destroyAllWindows()
    pipeline.close()
    print("Pipeline closed.")

Webcam opened. Press 'q' inside the window to quit.
Pipeline closed.


---
## 10 · (Optional) Train the Temporal Classifier on IPN Hand Features

Extracts InceptionV4 features from IPN Hand videos and trains the Transformer.  
Saves a checkpoint loadable in Cell 7. Update `VIDEO_DIR` and `ANNOT_CSV` to your paths.

In [12]:
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd

class IPNFeatureDataset(Dataset):
    VIDEO_DIR = Path('/root/.cache/kagglehub/datasets/soumicksarker/ipn-hand-dataset/versions/7/videos/videos')
    ANNOT_CSV = Path('/root/.cache/kagglehub/datasets/soumicksarker/ipn-hand-dataset/versions/7/metadata.csv')

    def __init__(self, spatial_ext, seq_len=Cfg.SEQ_LEN):
        self.spatial  = spatial_ext
        self.seq_len  = seq_len
        self.hands    = build_hands_tracker()
        self._ts      = 0
        meta          = pd.read_csv(self.ANNOT_CSV)
        self.records  = meta[['video_name', 'id', 'start_frame', 'end_frame']].dropna()
        print(f"IPNFeatureDataset: {len(self.records)} clips")

    def _next_ts(self):
        self._ts += 33
        return self._ts

    def _extract_sequence(self, video_path, start, end):
        cap        = cv2.VideoCapture(str(video_path))
        feats      = []
        total      = max(1, int(end) - int(start))
        sample_idx = set(int(start) + i * total // self.seq_len for i in range(self.seq_len))
        frame_no   = 0
        while cap.isOpened() and len(feats) < self.seq_len:
            ret, frame = cap.read()
            if not ret:
                break
            frame_no += 1
            if frame_no not in sample_idx:
                continue
            rgb      = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
            results  = self.hands.detect_for_video(mp_image, self._next_ts())
            crop     = get_hand_bbox(frame, results.hand_landmarks[0])[0] if results.hand_landmarks else frame
            feat     = self.spatial.encode(crop)
            if feat is not None:
                feats.append(feat)
        cap.release()
        while len(feats) < self.seq_len:
            feats.append(torch.zeros(Cfg.FEATURE_DIM))
        return torch.stack(feats[:self.seq_len])

    def __len__(self):  return len(self.records)
    def __getitem__(self, idx):
        row   = self.records.iloc[idx]
        path  = self.VIDEO_DIR / (str(row['video_name']) + '.avi')
        feats = self._extract_sequence(path, row['start_frame'], row['end_frame'])
        return feats, int(row['id']) - 1


def train_temporal_classifier(
    spatial_ext, classifier,
    epochs=20, batch_size=8, lr=1e-4,
    save_path='checkpoints/best_model.pt',
):
    Path(save_path).parent.mkdir(parents=True, exist_ok=True)
    loader    = DataLoader(IPNFeatureDataset(spatial_ext), batch_size=batch_size, shuffle=True, num_workers=2)
    optimizer = optim.AdamW(classifier.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    best_loss = float('inf')
    classifier.train()

    for epoch in range(1, epochs + 1):
        total_loss, correct, total = 0.0, 0, 0
        for feats, labels in loader:
            feats, labels = feats.to(Cfg.DEVICE), labels.to(Cfg.DEVICE)
            optimizer.zero_grad()
            loss = criterion(classifier(feats), labels)
            loss.backward()
            nn.utils.clip_grad_norm_(classifier.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item() * feats.size(0)
            correct    += (classifier(feats).argmax(1) == labels).sum().item()
            total      += feats.size(0)
        scheduler.step()
        print(f'Epoch {epoch:>3}/{epochs}  loss={total_loss/total:.4f}  acc={correct/total:.2%}')
        if total_loss / total < best_loss:
            best_loss = total_loss / total
            torch.save({'temporal_classifier': classifier.state_dict()}, save_path)
            print(f'  checkmark Saved to {save_path}')

    classifier.eval()
    print('Training complete.')


# Uncomment to train:
# train_temporal_classifier(spatial_extractor, temporal_classifier)